# Module `dynamic_costs.py`

Ce notebook explique la logique de cout dynamique appliquee aux vraies routes du graphe residuel.

Important : on ne modifie pas directement les aretes virtuelles du graphe complete. On modifie les vraies aretes, puis le generateur relance Dijkstra pour recalculer la completion metrique.

## Idee generale

Une route a trois niveaux de cout :

1. `base_cost` : cout minimal physique de la route.
2. `static_cost` : cout de base + surcout statique eventuel.
3. `dynamic_cost` : cout courant, qui evolue a chaque tour.

La dynamique doit rester realiste :

- elle peut augmenter ou diminuer temporairement le cout ;
- elle ne peut jamais descendre sous le cout de base ;
- elle ne peut pas monter indefiniment ;
- plus elle est haute, plus elle a tendance a revenir vers le cout statique.

In [ ]:
import sys
from pathlib import Path
import random

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.dynamic_costs import (
    dynamic_multiplier,
    initialize_dynamic_edge_costs,
    refresh_dynamic_edge_costs,
    sample_dynamic_edge_cost,
)
from cesipath.models import EdgeAttributes, EdgeStatus

## Tirage dynamique d'une seule arete

La fonction `sample_dynamic_edge_cost` prend :

- `edge` : l'arete reelle concernee ;
- `previous_cost` : le cout dynamique precedent ;
- `sigma` : amplitude de la variation aleatoire ;
- `mean_reversion_strength` : force de rappel vers le cout statique ;
- `max_multiplier` : borne haute relative au cout statique.

Le centre du prochain tirage n'est pas exactement le cout precedent. Il est corrige par un retour vers le cout statique :

```text
moyenne = previous_cost - alpha * (previous_cost - static_cost)
```

Si `previous_cost` est tres haut, la moyenne redescend. Cela evite les derives absurdes.

In [ ]:
edge = EdgeAttributes(base_cost=10.0, status=EdgeStatus.SURCHARGED, static_surcharge=0.25)
rng = random.Random(42)
previous_cost = None

print("Cout de base :", edge.base_cost)
print("Cout statique :", edge.static_cost)
print()

for step in range(1, 9):
    new_cost = sample_dynamic_edge_cost(
        edge,
        previous_cost=previous_cost,
        rng=rng,
        sigma=0.18,
        mean_reversion_strength=0.35,
        max_multiplier=1.8,
    )
    coefficient = dynamic_multiplier(edge, new_cost)
    print(f"Tour {step} -> cout={new_cost} ; coefficient={coefficient}")
    previous_cost = new_cost

## Comment lire le retour ?

Le coefficient affiche le rapport :

```text
coefficient = dynamic_cost / static_cost
```

Un coefficient inferieur a `1` signifie que la dynamique a reduit le surcout statique, mais le cout reste toujours au-dessus du cout de base.

Un coefficient superieur a `1` signifie que la route est temporairement plus couteuse que son cout statique.

Le rappel vers le statique evite qu'une route reste durablement dans des couts trop eleves.

## Mise a jour d'un petit reseau

A l'echelle du graphe, on met a jour les vraies aretes une par une.

`initialize_dynamic_edge_costs` initialise chaque route autorisee a son cout statique.

`refresh_dynamic_edge_costs` calcule ensuite un nouveau cout dynamique pour chaque route autorisee.

Dans le generateur complet, cette etape est suivie d'un recalcul Dijkstra pour reconstruire le graphe complete.

In [ ]:
edges = {
    (0, 1): EdgeAttributes(base_cost=10.0, status=EdgeStatus.SURCHARGED, static_surcharge=0.25),
    (1, 2): EdgeAttributes(base_cost=8.0),
}
network_rng = random.Random(7)
current_costs = initialize_dynamic_edge_costs(edges)
print("Etat initial :", current_costs)

current_costs = refresh_dynamic_edge_costs(
    edges,
    current_costs,
    rng=network_rng,
    sigma=0.18,
    mean_reversion_strength=0.35,
    max_multiplier=1.8,
)
print("Apres un deplacement :", current_costs)

## Pourquoi ce choix d'implementation ?

On combine deux protections :

- un frein progressif, plus naturel qu'une simple coupure brutale ;
- une borne haute, utile comme securite mathematique.

Cela permet de garder une dynamique visible, sans produire des couts incoherents ou impossibles a expliquer.